# BertDiffused — Colab Training Pipeline

End-to-end notebook for training BertMoEDiffusion on Google Colab:
1. **Setup** — Install deps, mount Drive, clone repo
2. **ETL** — Download LM1B, clean, deduplicate, tokenize, shard to Parquet
3. **Train** — LoRA + MoE fine-tuning with MLflow tracking
4. **Monitor** — Live loss curves, eval BPD, parameter breakdown
5. **Save** — Only LoRA adapters + final merged model to Drive (space-efficient)

> **Drive space strategy**: We save only LoRA adapters (~2.5 MB) during training 
> and one final merged model at the end. Intermediate full checkpoints are kept 
> only on the Colab VM (ephemeral) — not synced to Drive.

## 0. GPU Check & Environment

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available — go to Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive (small footprint)
DRIVE_OUTPUT = "/content/drive/MyDrive/BertDiffused"
!mkdir -p {DRIVE_OUTPUT}/lora_adapters
!mkdir -p {DRIVE_OUTPUT}/final_model
!mkdir -p {DRIVE_OUTPUT}/best_model
!mkdir -p {DRIVE_OUTPUT}/mlruns
!mkdir -p {DRIVE_OUTPUT}/datasets_cache
print(f"Drive output dir: {DRIVE_OUTPUT}")


In [ ]:
import os
os.chdir('/content/drive/MyDrive/BertDiffused')
print(f"Working dir: {os.getcwd()}")

# Point HuggingFace cache to Drive so datasets are never re-downloaded
_hf_cache = f"{DRIVE_OUTPUT}/datasets_cache"
os.environ["HF_DATASETS_CACHE"] = _hf_cache
os.environ["HF_HOME"] = _hf_cache
print(f"HF datasets cache: {_hf_cache}")


## 2. Install Dependencies

In [ ]:
# Pin datasets<3.0.0 before installing requirements.
# lm1b uses a legacy dataset script (lm1b.py) which datasets v3+ dropped support for.
!pip install -q "datasets>=2.19.0,<3.0.0"
!pip install -q -r /content/drive/MyDrive/BertDiffused/requirements.txt
# Re-pin fsspec after datasets (datasets<3 downgrades it, breaking gcsfs on Colab)
!pip install -q "fsspec>=2025.3.0"


In [ ]:
import mlflow
import mlflow.pytorch
import numpy as np
import yaml
import logging
import math
import time as time_module
from pathlib import Path
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

print(f"MLflow version: {mlflow.__version__}")

## 3. Load Configuration

In [ ]:
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Colab overrides ────────────────────────────────────────────────────────
# Reduce batch size for Colab GPU memory
cfg["training"]["batch_size"] = 32
cfg["training"]["gradient_accumulation_steps"] = 16  # effective batch = 512
cfg["training"]["max_steps"] = 50000                 # shorter run for Colab
cfg["training"]["save_steps"] = 10000
cfg["training"]["eval_steps"] = 5000
cfg["training"]["log_steps"] = 50
cfg["training"]["output_dir"] = "/content/checkpoints"  # ephemeral (Colab VM)

# ETL: limit samples for Colab (set to None for full dataset)
cfg["etl"]["max_samples"] = 500000   # 500K samples — adjust based on time budget
cfg["etl"]["output_dir"] = "/content/data_processed"
cfg["etl"]["num_proc"] = 2

# MLflow: store on Drive for persistence
cfg["mlflow"]["tracking_uri"] = f"{DRIVE_OUTPUT}/mlruns"

# LoRA: enabled by default (parameter-efficient)
cfg["model"]["lora"]["enabled"] = True

print("Config loaded with Colab overrides.")
print(f"  Batch: {cfg['training']['batch_size']} x {cfg['training']['gradient_accumulation_steps']} = {cfg['training']['batch_size'] * cfg['training']['gradient_accumulation_steps']}")
print(f"  Max steps: {cfg['training']['max_steps']:,}")
print(f"  LoRA: rank={cfg['model']['lora']['rank']}, alpha={cfg['model']['lora']['alpha']}")
print(f"  ETL samples: {cfg['etl']['max_samples']:,}")

## 4. Download & Process Data (ETL Pipeline)

**Step 1**: Download the LM1B dataset from HuggingFace Hub  
**Step 2**: Run the ETL pipeline — clean, deduplicate, tokenize, shard to Parquet  

Output: Parquet shards in `/content/data_processed/{train,test}/`

In [ ]:
from datasets import load_dataset

dataset_name = cfg["training"]["dataset"]
max_samples = cfg["etl"]["max_samples"]

print(f"Downloading dataset: {dataset_name}")
print(f"Max samples: {max_samples:,}\n")

# Download train split (cached to Drive — skip download on subsequent runs)
print("Downloading TRAIN split...")
train_raw = load_dataset(dataset_name, split="train", trust_remote_code=True,
                         cache_dir=f"{DRIVE_OUTPUT}/datasets_cache")
print(f"  Train: {len(train_raw):,} total samples")

# Download test split
print("Downloading TEST split...")
test_raw = load_dataset(dataset_name, split="test", trust_remote_code=True,
                        cache_dir=f"{DRIVE_OUTPUT}/datasets_cache")
print(f"  Test:  {len(test_raw):,} total samples")

# Preview a few examples
print("\n" + "="*60)
print("Sample data:")
print("="*60)
for i in range(min(5, len(train_raw))):
    text = train_raw[i]["text"][:120]
    print(f"  [{i}] {text}{'...' if len(train_raw[i]['text']) > 120 else ''}")

# Check Drive cache size
!du -sh {DRIVE_OUTPUT}/datasets_cache/ 2>/dev/null || echo "Cache dir not found"


In [ ]:
from transformers import AutoTokenizer
from data.etl import DataETL

tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["backbone"])
mask_token_id = tokenizer.mask_token_id

# Setup MLflow for ETL tracking
mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
mlflow.set_experiment(cfg["mlflow"]["experiment_name"])

etl = DataETL(cfg, tokenizer, output_dir=cfg["etl"]["output_dir"])

with mlflow.start_run(run_name="colab-etl"):
    mlflow.set_tag("pipeline_stage", "etl")
    mlflow.set_tag("runtime", "colab")

    print("\n" + "="*60)
    print("ETL: Processing TRAIN split")
    print("="*60)
    etl.run(split="train")

    print("\n" + "="*60)
    print("ETL: Processing TEST split")
    print("="*60)
    # Use smaller test set
    cfg_test = cfg.copy()
    etl_test = DataETL(cfg_test, tokenizer, output_dir=cfg["etl"]["output_dir"])
    etl_test.run(split="test")

print("\nETL Stats:")
for k, v in etl.stats.items():
    print(f"  {k}: {v:,.2f}" if isinstance(v, float) else f"  {k}: {v:,}")

In [ ]:
# Verify ETL output
import glob

train_shards = sorted(glob.glob(f"{cfg['etl']['output_dir']}/train/shard-*.parquet"))
test_shards = sorted(glob.glob(f"{cfg['etl']['output_dir']}/test/shard-*.parquet"))

print(f"Train shards: {len(train_shards)}")
print(f"Test shards:  {len(test_shards)}")

# Check disk usage
!du -sh {cfg['etl']['output_dir']}/train/ {cfg['etl']['output_dir']}/test/ 2>/dev/null || echo "Shards not found"

## 5. Build Model (LoRA + MoE)

In [ ]:
from model import BertMoEDiffusion, LogLinearNoiseSchedule

moe_cfg = cfg["model"]["moe"]
lora_cfg = cfg["model"]["lora"]

model = BertMoEDiffusion(
    bert_model_name=cfg["model"]["backbone"],
    moe_layers=moe_cfg["moe_layers"],
    num_experts=moe_cfg["num_experts"],
    num_experts_per_token=moe_cfg["num_experts_per_token"],
    expert_hidden_multiplier=moe_cfg["expert_hidden_multiplier"],
    router_jitter=moe_cfg["router_jitter"],
    router_z_loss_coef=moe_cfg["router_z_loss_coef"],
    router_aux_loss_coef=moe_cfg["router_aux_loss_coef"],
    time_embed_dim=cfg["model"]["time_embed_dim"],
    use_time_conditioning=cfg["model"]["use_time_conditioning"],
    dropout=cfg["model"]["dropout"],
    lora_enabled=lora_cfg["enabled"],
    lora_rank=lora_cfg["rank"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    lora_target_modules=lora_cfg["target_modules"],
)
model.set_mask_token_id(mask_token_id)

device = torch.device("cuda")
model.to(device)

# Parameter summary
ps = model.trainable_parameters_summary()
print("\n" + "="*50)
print("MODEL PARAMETER SUMMARY")
print("="*50)
print(f"Total params:     {ps['total']:>12,}")
print(f"Trainable params: {ps['trainable']:>12,}  ({ps['trainable_pct']:.1f}%)")
print(f"Frozen params:    {ps['frozen']:>12,}")
print(f"├── LoRA params:  {ps['lora']:>12,}")
print(f"└── MoE params:   {ps['moe']:>12,}")
print(f"\nGPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## 6. Prepare Data Loaders

In [ ]:
from torch.utils.data import DataLoader
from data.etl import ProcessedParquetDataset
from data.lm1b_dataset import LM1BDataset

train_etl_dir = Path(cfg["etl"]["output_dir"]) / "train"
eval_etl_dir = Path(cfg["etl"]["output_dir"]) / "test"

if train_etl_dir.exists() and any(train_etl_dir.glob("shard-*.parquet")):
    print("Using preprocessed ETL Parquet shards")
    train_dataset = ProcessedParquetDataset(train_etl_dir)
    eval_dataset = ProcessedParquetDataset(eval_etl_dir)
else:
    print("No ETL shards found — falling back to raw on-the-fly tokenization")
    train_dataset = LM1BDataset(split="train", tokenizer=tokenizer, max_seq_len=cfg["model"]["max_seq_len"])
    eval_dataset = LM1BDataset(split="test", tokenizer=tokenizer, max_seq_len=cfg["model"]["max_seq_len"])

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg["training"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=cfg["evaluation"]["eval_batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"Train: {len(train_dataset):,} samples, {len(train_loader):,} batches")
print(f"Eval:  {len(eval_dataset):,} samples")

## 7. Training Loop with MLflow Tracking

**Storage strategy** (Drive-friendly):
- Intermediate checkpoints → Colab VM only (ephemeral, overwritten each save)
- LoRA adapters → Drive (~2.5 MB per save)
- Final merged model → Drive (one copy)
- MLflow tracking DB → Drive (persistent across sessions)

In [ ]:
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup

# ── Loss function (from train.py) ──────────────────────────────────────────
def compute_mdlm_loss(logits, input_ids, z_t, t, mask_token_id, time_eps=1e-4):
    B, L, V = logits.shape
    weights = 1.0 / t.clamp(min=time_eps)
    is_masked = (z_t == mask_token_id)
    ce = F.cross_entropy(
        logits.reshape(B * L, V), input_ids.reshape(B * L), reduction='none'
    ).reshape(B, L)
    ce = ce * is_masked.float()
    loss_per_seq = ce.sum(-1)
    return (weights * loss_per_seq).mean()

# ── Eval BPD ───────────────────────────────────────────────────────────────
# max_batches=None  → run on the full dataloader (final test evaluation)
# max_batches=20    → quick estimate used during training
@torch.no_grad()
def evaluate_bpd(model, dataloader, noise_schedule, mask_token_id, device,
                 num_eval_steps=50, time_eps=1e-4, max_batches=20):
    model.eval()
    total_loss, total_tokens, n_batches = 0.0, 0, 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        B, L = input_ids.shape
        batch_loss = 0.0
        for _ in range(num_eval_steps):
            t = noise_schedule.sample_t(B, device, low_discrepancy=True)
            z_t = noise_schedule.noise_sequence(input_ids, t, mask_token_id)
            logits = model(z_t, t, attention_mask=attention_mask)
            loss = compute_mdlm_loss(logits, input_ids, z_t, t, mask_token_id, time_eps)
            batch_loss += loss.item()
        total_loss += batch_loss / num_eval_steps
        total_tokens += B * L
        n_batches += 1
        if max_batches is not None and n_batches >= max_batches:
            break
    avg_nll = total_loss / n_batches
    return avg_nll / (math.log(2) * (total_tokens / (n_batches * input_ids.shape[0])))

print("Loss and eval functions ready.")


In [ ]:
# ── Setup optimizer, scheduler, scaler ─────────────────────────────────────
noise_schedule = LogLinearNoiseSchedule()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=cfg["training"]["learning_rate"],
    betas=(cfg["training"]["adam_beta1"], cfg["training"]["adam_beta2"]),
    eps=cfg["training"]["adam_epsilon"],
    weight_decay=cfg["training"]["weight_decay"],
)
# Scheduler counts optimizer steps (actual weight updates), not global gradient-accumulation steps
_accum = cfg["training"]["gradient_accumulation_steps"]
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg["training"]["warmup_steps"] // _accum,
    num_training_steps=cfg["training"]["max_steps"] // _accum,
)
scaler = torch.amp.GradScaler("cuda", enabled=cfg["training"]["fp16"],
                               init_scale=1024, growth_interval=2000)

print(f"Optimizer: AdamW, LR={cfg['training']['learning_rate']}, {len(trainable_params)} param groups")
print(f"Scheduler: linear warmup ({cfg['training']['warmup_steps']} steps) + decay")
print(f"GradScaler: init_scale=1024, growth_interval=2000")


In [ ]:

# ── Anomaly Watcher ─────────────────────────────────────────────────────────
# Monitors training health each step and raises TrainingAnomalyError on:
#   1. NaN / Inf loss
#   2. Loss spike  — current loss > spike_factor × recent average
#   3. Loss plateau — no improvement over plateau_window steps
#   4. MoE aux runaway — aux loss exceeds moe_aux_threshold
#   5. Grad norm explosion — unclipped norm exceeds grad_norm_threshold

class TrainingAnomalyError(RuntimeError):
    pass

class AnomalyWatcher:
    def __init__(
        self,
        spike_factor: float = 5.0,        # flag if loss jumps >5× recent avg
        plateau_window: int = 5000,        # steps with no improvement = plateau
        plateau_min_delta: float = 0.01,   # min fractional improvement to count
        moe_aux_threshold: float = 1.0,    # aux loss above this = runaway
        grad_norm_threshold: float = 500.0,
    ):
        self.spike_factor = spike_factor
        self.plateau_window = plateau_window
        self.plateau_min_delta = plateau_min_delta
        self.moe_aux_threshold = moe_aux_threshold
        self.grad_norm_threshold = grad_norm_threshold

        self._loss_history = []      # rolling window for spike/plateau detection
        self._best_loss = float("inf")
        self._steps_since_improvement = 0

    def check(self, step: int, elbo_loss: float, moe_aux: float, grad_norm: float):
        """Call once per log interval. Raises TrainingAnomalyError on detection."""

        # 1. NaN / Inf
        if not math.isfinite(elbo_loss):
            raise TrainingAnomalyError(
                f"[Step {step}] ELBO loss is {elbo_loss} — NaN/Inf detected. "
                "Possible causes: fp16 overflow, exploding gradients, bad batch."
            )

        # 2. Spike detection (need at least 5 history points)
        self._loss_history.append(elbo_loss)
        if len(self._loss_history) > 50:
            self._loss_history.pop(0)
        if len(self._loss_history) >= 5:
            recent_avg = sum(self._loss_history[:-1]) / (len(self._loss_history) - 1)
            if elbo_loss > self.spike_factor * recent_avg:
                raise TrainingAnomalyError(
                    f"[Step {step}] Loss spike: {elbo_loss:.2f} is {elbo_loss/recent_avg:.1f}× "
                    f"the recent average ({recent_avg:.2f}). Possible instability."
                )

        # 3. Plateau detection
        if elbo_loss < self._best_loss * (1.0 - self.plateau_min_delta):
            self._best_loss = elbo_loss
            self._steps_since_improvement = 0
        else:
            self._steps_since_improvement += 1
        if self._steps_since_improvement >= self.plateau_window:
            raise TrainingAnomalyError(
                f"[Step {step}] Loss plateau: no improvement for {self.plateau_window} steps "
                f"(best={self._best_loss:.4f}, current={elbo_loss:.4f}). Model may be stuck."
            )

        # 4. MoE aux runaway
        if math.isfinite(moe_aux) and moe_aux > self.moe_aux_threshold:
            raise TrainingAnomalyError(
                f"[Step {step}] MoE aux loss runaway: {moe_aux:.4f} > threshold {self.moe_aux_threshold}. "
                "Expert collapse likely — increase router_aux_loss_coef."
            )

        # 5. Gradient norm explosion
        if math.isfinite(grad_norm) and grad_norm > self.grad_norm_threshold:
            raise TrainingAnomalyError(
                f"[Step {step}] Gradient norm explosion: {grad_norm:.1f} > threshold {self.grad_norm_threshold}. "
                "After clipping — check learning rate or model init."
            )

watcher = AnomalyWatcher(
    spike_factor=5.0,
    plateau_window=5000,
    # moe_aux_loss is summed across all MoE layers (5 in default config).
    # With aux_loss_coef=0.1 and 8 experts, balanced routing = ~0.4 total.
    # Threshold = 3.0 gives ~7x headroom over balanced before alarming.
    moe_aux_threshold=3.0,
    grad_norm_threshold=2000.0,  # pre-clip norm; actual update is clipped to max_grad_norm=1.0
)
print("AnomalyWatcher ready.")
print("  Monitors: NaN/Inf loss | loss spike (5×) | plateau (5k steps) | MoE aux runaway | grad norm explosion")


In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────

# Tracking lists for live plots
history = {"step": [], "elbo_loss": [], "moe_aux": [], "lr": [], "bpd": [], "bpd_step": []}

accum_steps = cfg["training"]["gradient_accumulation_steps"]
max_steps = cfg["training"]["max_steps"]
log_steps = cfg["training"]["log_steps"]
eval_steps = cfg["training"]["eval_steps"]
save_steps = cfg["training"]["save_steps"]
time_eps = float(cfg["diffusion"]["time_eps"])   # cast: YAML may parse 1e-4 as string
use_fp16 = cfg["training"]["fp16"]
output_dir = Path(cfg["training"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# MLflow run
mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
mlflow.set_experiment(cfg["mlflow"]["experiment_name"])
mlflow_run = mlflow.start_run(run_name="colab-training")

# Flatten and log config
def flatten_cfg(cfg, prefix=""):
    flat = {}
    for k, v in cfg.items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            flat.update(flatten_cfg(v, key))
        else:
            flat[key] = v
    return flat

mlflow.log_params(flatten_cfg(cfg))
mlflow.set_tags({"runtime": "colab", "lora_enabled": str(lora_cfg["enabled"]),
                 "gpu": torch.cuda.get_device_name(0)})
mlflow.log_metrics({f"model/{k}": v for k, v in ps.items()})

# ── Loop ───────────────────────────────────────────────────────────────────
model.train()
optimizer.zero_grad()
running_loss = 0.0
running_moe_loss = 0.0
running_grad_norm = 0.0
global_step = 0
best_bpd = float("inf")
best_bpd_step = 0
train_iter = iter(train_loader)
t_start = time_module.time()

print(f"Training started — {max_steps:,} steps, device={device}")
print(f"Checkpoints (VM): {output_dir}")
print(f"LoRA adapters (Drive): {DRIVE_OUTPUT}/lora_adapters/")
print(f"Best model (Drive):    {DRIVE_OUTPUT}/best_model/")
print("-" * 60)

def _emergency_checkpoint(reason: str):
    """Save model + LoRA to disk immediately when an anomaly is detected."""
    ckpt_path = output_dir / "emergency_checkpoint.pt"
    torch.save(
        {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
         "scheduler": scheduler.state_dict(), "global_step": global_step,
         "config": cfg, "stop_reason": reason},
        ckpt_path,
    )
    lora_state = {k: v.cpu() for k, v in model.state_dict().items()
                  if "lora_A" in k or "lora_B" in k}
    lora_path = f"{DRIVE_OUTPUT}/lora_adapters/lora_emergency_step_{global_step}.pt"
    torch.save({"lora_state_dict": lora_state, "global_step": global_step,
                "stop_reason": reason, "config": cfg}, lora_path)
    print(f"  Emergency checkpoint saved → {ckpt_path}")
    print(f"  Emergency LoRA saved      → {lora_path}")

try:
    while global_step < max_steps:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        B, L = input_ids.shape

        t = noise_schedule.sample_t(B, device, low_discrepancy=True)
        z_t = noise_schedule.noise_sequence(input_ids, t, mask_token_id)

        with torch.amp.autocast("cuda", enabled=use_fp16):
            logits = model(z_t, t, attention_mask=attention_mask)
            elbo_loss = compute_mdlm_loss(logits, input_ids, z_t, t, mask_token_id, time_eps)
            moe_aux = model.moe_aux_loss
            total_loss = (elbo_loss + moe_aux) / accum_steps

        scaler.scale(total_loss).backward()
        running_loss += elbo_loss.item()
        running_moe_loss += moe_aux.item() if isinstance(moe_aux, torch.Tensor) else moe_aux

        if (global_step + 1) % accum_steps == 0 or global_step == max_steps - 1:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), cfg["training"]["max_grad_norm"]
            ).item()
            if math.isfinite(grad_norm):
                running_grad_norm += grad_norm
            _scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            # Only step scheduler when optimizer actually updated weights
            if scaler.get_scale() == _scale_before:
                scheduler.step()
            optimizer.zero_grad()

        global_step += 1

        # ── Logging ────────────────────────────────────────────────────────
        if global_step % log_steps == 0:
            avg_loss = running_loss / log_steps
            avg_moe = running_moe_loss / log_steps
            avg_grad_norm = running_grad_norm / max(1, log_steps // accum_steps)
            lr = scheduler.get_last_lr()[0]
            elapsed = time_module.time() - t_start
            steps_per_sec = global_step / elapsed
            eta_min = (max_steps - global_step) / steps_per_sec / 60

            history["step"].append(global_step)
            history["elbo_loss"].append(avg_loss)
            history["moe_aux"].append(avg_moe)
            history["lr"].append(lr)

            mlflow.log_metrics(
                {"train/elbo_loss": avg_loss, "train/moe_aux_loss": avg_moe,
                 "train/learning_rate": lr, "train/steps_per_sec": steps_per_sec,
                 "train/grad_norm": avg_grad_norm},
                step=global_step,
            )

            print(
                f"Step {global_step:6d}/{max_steps} | ELBO {avg_loss:.4f} | "
                f"MoE {avg_moe:.4f} | GradNorm {avg_grad_norm:.1f} | LR {lr:.2e} | "
                f"{steps_per_sec:.1f} steps/s | ETA {eta_min:.0f}m"
            )
            running_loss = 0.0
            running_moe_loss = 0.0
            running_grad_norm = 0.0

            # ── Anomaly check (at every log interval) ─────────────────────
            watcher.check(global_step, avg_loss, avg_moe, avg_grad_norm)

        # ── Evaluation + best-model checkpoint ─────────────────────────────
        if global_step % eval_steps == 0:
            bpd = evaluate_bpd(model, eval_loader, noise_schedule, mask_token_id, device, time_eps=time_eps)
            history["bpd"].append(bpd)
            history["bpd_step"].append(global_step)
            mlflow.log_metric("eval/bpd", bpd, step=global_step)
            print(f"  >>> Eval BPD: {bpd:.4f}")

            if bpd < best_bpd:
                best_bpd = bpd
                best_bpd_step = global_step
                best_path = f"{DRIVE_OUTPUT}/best_model/best_model.pt"
                torch.save(
                    {"model": model.state_dict(), "config": cfg,
                     "global_step": global_step, "bpd": bpd},
                    best_path,
                )
                lora_best = {k: v.cpu() for k, v in model.state_dict().items()
                             if "lora_A" in k or "lora_B" in k}
                torch.save({"lora_state_dict": lora_best, "global_step": global_step,
                            "bpd": bpd, "config": cfg},
                           f"{DRIVE_OUTPUT}/lora_adapters/lora_best.pt")
                mlflow.log_metric("eval/best_bpd", bpd, step=global_step)
                print(f"  *** New best BPD {bpd:.4f} at step {global_step} — saved to Drive")

            model.train()

        # ── Checkpoint (VM only — overwrite to save space) ─────────────────
        if global_step % save_steps == 0:
            ckpt_path = output_dir / "latest_checkpoint.pt"
            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "global_step": global_step,
                 "config": cfg},
                ckpt_path,
            )
            print(f"  Checkpoint saved (VM): {ckpt_path}")

            lora_state = {k: v.cpu() for k, v in model.state_dict().items()
                          if "lora_A" in k or "lora_B" in k}
            lora_drive_path = f"{DRIVE_OUTPUT}/lora_adapters/lora_step_{global_step}.pt"
            torch.save({"lora_state_dict": lora_state, "global_step": global_step,
                        "config": cfg}, lora_drive_path)
            print(f"  LoRA adapters saved (Drive): {lora_drive_path}")

except TrainingAnomalyError as e:
    print(f"\n{'='*60}")
    print(f"  TRAINING STOPPED — ANOMALY DETECTED")
    print(f"{'='*60}")
    print(f"  {e}")
    print(f"{'='*60}")
    mlflow.set_tag("stop_reason", str(e))
    _emergency_checkpoint(str(e))

except KeyboardInterrupt:
    print("\nTraining interrupted by user.")

print(f"\nTraining finished at step {global_step}.")
print(f"Best BPD: {best_bpd:.4f} at step {best_bpd_step}")


## 8. Training Monitoring — Live Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("BertDiffused Training Monitor", fontsize=14, fontweight="bold")

# ELBO Loss
axes[0, 0].plot(history["step"], history["elbo_loss"], "b-", linewidth=0.8)
axes[0, 0].set_title("ELBO Loss")
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(True, alpha=0.3)

# MoE Auxiliary Loss
axes[0, 1].plot(history["step"], history["moe_aux"], "r-", linewidth=0.8)
axes[0, 1].set_title("MoE Auxiliary Loss")
axes[0, 1].set_xlabel("Step")
axes[0, 1].set_ylabel("Aux Loss")
axes[0, 1].grid(True, alpha=0.3)

# Learning Rate
axes[1, 0].plot(history["step"], history["lr"], "g-", linewidth=0.8)
axes[1, 0].set_title("Learning Rate")
axes[1, 0].set_xlabel("Step")
axes[1, 0].set_ylabel("LR")
axes[1, 0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[1, 0].grid(True, alpha=0.3)

# Eval BPD
if history["bpd"]:
    axes[1, 1].plot(history["bpd_step"], history["bpd"], "mo-", linewidth=1.2, markersize=6)
    axes[1, 1].set_title("Eval BPD (bits-per-dim)")
    axes[1, 1].set_xlabel("Step")
    axes[1, 1].set_ylabel("BPD")
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, "No eval data yet", ha='center', va='center', fontsize=12)
    axes[1, 1].set_title("Eval BPD")

plt.tight_layout()
plt.savefig(f"{DRIVE_OUTPUT}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {DRIVE_OUTPUT}/training_curves.png")

## 9. Save Best Model to Drive

Loads the **best checkpoint** (lowest eval BPD) saved during training, merges LoRA weights into base BERT, and saves a single inference-ready model to Drive.


In [ ]:
# ── Load best checkpoint (lowest eval BPD) ────────────────────────────────
best_ckpt_path = f"{DRIVE_OUTPUT}/best_model/best_model.pt"
if os.path.exists(best_ckpt_path):
    best_ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(best_ckpt["model"])
    _saved_step = best_ckpt["global_step"]
    _saved_bpd  = best_ckpt["bpd"]
    print(f"Loaded best model: step={_saved_step:,}, train-eval BPD={_saved_bpd:.4f}")
else:
    print("WARNING: No best_model.pt found on Drive — using current model weights.")
    _saved_step = global_step
    _saved_bpd  = best_bpd

# ── Full test-set evaluation (optimized for speed) ─────────────────────────
print(f"\nRunning fast evaluation on test set ({len(eval_dataset):,} samples)...")
test_bpd = evaluate_bpd(
    model, eval_loader, noise_schedule, mask_token_id, device,
    num_eval_steps=20, time_eps=time_eps,
    max_batches=100,   # estimate from first 100 batches (~2-3 min instead of hours)
)
print(f"  Test BPD (estimate):  {test_bpd:.4f}")
mlflow.log_metric("test/bpd_estimate", test_bpd, step=_saved_step)
mlflow.set_tag("test_bpd_estimate", f"{test_bpd:.4f}")

# Save best LoRA adapters (before merge)
lora_best_save = {k: v.cpu() for k, v in model.state_dict().items()
                  if "lora_A" in k or "lora_B" in k}
lora_final_path = f"{DRIVE_OUTPUT}/lora_adapters/lora_final.pt"
torch.save({"lora_state_dict": lora_best_save, "global_step": _saved_step,
            "bpd": _saved_bpd, "test_bpd": test_bpd, "config": cfg}, lora_final_path)
print(f"\nFinal LoRA adapters: {lora_final_path}")
mlflow.log_artifact(lora_final_path, artifact_path="lora_adapters")

# Merge LoRA into base weights (zero inference overhead)
model.merge_lora()
print("LoRA weights merged into base model.")

# Save merged model to Drive
final_path = f"{DRIVE_OUTPUT}/final_model/final_model.pt"
torch.save({"model": model.state_dict(), "config": cfg,
            "global_step": _saved_step, "bpd": _saved_bpd,
            "test_bpd": test_bpd}, final_path)
print(f"Final merged model: {final_path}")
print(f"\n{'='*50}")
print(f"  Best val BPD:     {_saved_bpd:.4f}  (step {_saved_step:,})")
print(f"  Test BPD (est.):  {test_bpd:.4f}  (100 batches)")
print(f"{'='*50}")

# Log to MLflow Model Registry
mlflow.log_artifact(final_path, artifact_path="final_model")
mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path="model",
    registered_model_name=cfg["mlflow"]["registered_model_name"],
    pip_requirements=["torch>=2.2.0", "transformers>=4.40.0", "mlflow>=2.14.0"],
)
print(f"Model registered in MLflow as '{cfg['mlflow']['registered_model_name']}'")

# End MLflow run
mlflow.end_run()
print("MLflow run ended.")

## 10. Drive Storage Summary

In [ ]:
print("Files saved to Google Drive:")
print("=" * 50)
!du -sh {DRIVE_OUTPUT}/*
print()
print("LoRA adapter snapshots:")
!ls -lh {DRIVE_OUTPUT}/lora_adapters/
print()
print("Final model:")
!ls -lh {DRIVE_OUTPUT}/final_model/

## 11. Resume Training (if Colab disconnects)

Run this cell to resume from the latest checkpoint on the VM,  
or reload LoRA adapters from Drive if the VM was recycled.

In [ ]:
# ── Resume helpers ─────────────────────────────────────────────────────────

def resume_from_vm_checkpoint(model, optimizer, scheduler, ckpt_path="/content/checkpoints/latest_checkpoint.pt"):
    """Resume from full checkpoint on Colab VM (if session is still alive)."""
    if not os.path.exists(ckpt_path):
        print(f"No VM checkpoint found at {ckpt_path}")
        return 0
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    step = ckpt["global_step"]
    print(f"Resumed from VM checkpoint at step {step:,}")
    return step

def resume_from_drive_lora(model, drive_output=DRIVE_OUTPUT):
    """Reload the latest LoRA adapters from Drive (if VM was recycled)."""
    import glob
    lora_files = sorted(glob.glob(f"{drive_output}/lora_adapters/lora_step_*.pt"))
    if not lora_files:
        print("No LoRA adapters found on Drive.")
        return 0
    latest = lora_files[-1]
    ckpt = torch.load(latest, map_location=device)
    # Load only LoRA params into model
    model_state = model.state_dict()
    model_state.update(ckpt["lora_state_dict"])
    model.load_state_dict(model_state)
    step = ckpt.get("global_step", 0)
    print(f"Loaded LoRA adapters from Drive: {latest} (step {step:,})")
    print("NOTE: Optimizer/scheduler state is lost — training resumes with fresh optimizer.")
    return step

# Uncomment one of these to resume:
# global_step = resume_from_vm_checkpoint(model, optimizer, scheduler)
# global_step = resume_from_drive_lora(model)

## 12. Smoke Test — Post-Training Validation

Runs 11 end-to-end tests on CPU with synthetic data to verify the model code is intact after training.  
Safe to re-run at any time — uses a separate minimal model instance, does not touch the trained model.


In [ ]:
# ── Smoke Test ──────────────────────────────────────────────────────────────
# Self-contained CPU test. Uses a minimal config (multiplier=2, 1 MoE layer).
# All variables prefixed with _ to avoid overwriting the trained model's state.

import math as _math
import traceback as _traceback
import torch as _torch
import torch.nn.functional as _F
from transformers import AutoTokenizer as _AutoTokenizer, get_linear_schedule_with_warmup as _get_sched

_VOCAB_SIZE = 30522
_SEQ_LEN = 32
_BATCH_SIZE = 2
_MAX_STEPS = 5
_DEVICE = _torch.device("cpu")

_cfg_smoke = {
    "model": {
        "backbone": "bert-base-uncased",
        "hidden_size": 768,
        "max_seq_len": _SEQ_LEN,
        "dropout": 0.1,
        "use_time_conditioning": True,
        "time_embed_dim": 128,
        "moe": {
            "num_experts": 4,
            "num_experts_per_token": 2,
            "moe_layers": [3],
            "expert_hidden_multiplier": 2,   # intentionally ≠ 4 → dimension guard tested
            "router_jitter": 0.01,
            "router_z_loss_coef": 0.001,
            "router_aux_loss_coef": 0.01,
        },
        "lora": {
            "enabled": True,
            "rank": 4,
            "alpha": 8.0,
            "dropout": 0.05,
            "target_modules": ["query", "key", "value"],
        },
    },
    "diffusion": {"time_eps": 1.0e-4},
    "training": {
        "learning_rate": 5e-5,
        "warmup_steps": 1,
        "max_steps": _MAX_STEPS,
        "max_grad_norm": 1.0,
        "gradient_accumulation_steps": 1,
        "fp16": False,
    },
}

def _section(title): print(f"\n{'='*60}\n  {title}\n{'='*60}")
def _ok(msg):        print(f"  [PASS] {msg}")
def _fail(msg, exc):
    print(f"  [FAIL] {msg}")
    _traceback.print_exc()
    raise RuntimeError(f"Smoke test failed: {msg}") from exc

_passed = 0

# 1. Imports
_section("1. Imports")
try:
    from model import BertMoEDiffusion as _BMD, LogLinearNoiseSchedule as _LNS
    _ok("model imports"); _passed += 1
except Exception as _e: _fail("model imports", _e)

# 2. Tokenizer
_section("2. Tokenizer")
try:
    _tok = _AutoTokenizer.from_pretrained("bert-base-uncased")
    _mask_id = _tok.mask_token_id
    _ok(f"tokenizer loaded, mask_token_id={_mask_id}"); _passed += 1
except Exception as _e: _fail("tokenizer", _e)

# 3. Model construction
_section("3. Model construction")
try:
    _mc, _moe_c, _lora_c = _cfg_smoke["model"], _cfg_smoke["model"]["moe"], _cfg_smoke["model"]["lora"]
    _sm = _BMD(
        bert_model_name=_mc["backbone"],
        moe_layers=_moe_c["moe_layers"],
        num_experts=_moe_c["num_experts"],
        num_experts_per_token=_moe_c["num_experts_per_token"],
        expert_hidden_multiplier=_moe_c["expert_hidden_multiplier"],
        router_jitter=_moe_c["router_jitter"],
        router_z_loss_coef=_moe_c["router_z_loss_coef"],
        router_aux_loss_coef=_moe_c["router_aux_loss_coef"],
        time_embed_dim=_mc["time_embed_dim"],
        use_time_conditioning=_mc["use_time_conditioning"],
        dropout=_mc["dropout"],
        lora_enabled=_lora_c["enabled"],
        lora_rank=_lora_c["rank"],
        lora_alpha=_lora_c["alpha"],
        lora_dropout=_lora_c["dropout"],
        lora_target_modules=_lora_c["target_modules"],
    )
    _sm.set_mask_token_id(_mask_id)
    _sm.to(_DEVICE)
    _ps = _sm.trainable_parameters_summary()
    _ok(f"model OK — {_ps['total']:,} total | {_ps['trainable']:,} trainable | {_ps['lora']:,} LoRA | {_ps['moe']:,} MoE")
    _passed += 1
except Exception as _e: _fail("model construction", _e)

# 4. Noise schedule
_section("4. Noise schedule")
try:
    _ns = _LNS()
    _t_test = _torch.tensor([0.1, 0.5, 0.9])
    assert _torch.allclose(_ns.alpha(_t_test), 1.0 - _t_test)
    _t_samp = _ns.sample_t(_BATCH_SIZE, _DEVICE, low_discrepancy=True)
    assert _t_samp.shape == (_BATCH_SIZE,)
    _ok(f"alpha(t)=1−t verified; sample_t shape OK"); _passed += 1
except Exception as _e: _fail("noise schedule", _e)

# 5. Forward pass
_section("5. Forward pass")
try:
    _ids = _torch.randint(1000, 20000, (_BATCH_SIZE, _SEQ_LEN))
    _attn = _torch.ones(_BATCH_SIZE, _SEQ_LEN, dtype=_torch.long)
    _t = _ns.sample_t(_BATCH_SIZE, _DEVICE)
    _zt = _ns.noise_sequence(_ids, _t, _mask_id)
    _sm.eval()
    with _torch.no_grad():
        _logits = _sm(_zt, _t, attention_mask=_attn)
    assert _logits.shape == (_BATCH_SIZE, _SEQ_LEN, _VOCAB_SIZE)
    _ok(f"logits shape: {_logits.shape}"); _passed += 1
except Exception as _e: _fail("forward pass", _e)

# 6. MDLM loss
_section("6. MDLM loss")
try:
    def _mdlm(lgts, ids, zt, t, mid, eps=1e-4):
        B, L, V = lgts.shape
        w = 1.0 / t.clamp(min=eps)
        ce = _F.cross_entropy(lgts.reshape(B*L,V), ids.reshape(B*L), reduction="none").reshape(B,L)
        return (w * (ce * (zt==mid).float()).sum(-1)).mean()
    _loss = _mdlm(_logits, _ids, _zt, _t, _mask_id)
    assert _loss.item() > 0 and _math.isfinite(_loss.item())
    _ok(f"MDLM loss = {_loss.item():.4f}"); _passed += 1
except Exception as _e: _fail("MDLM loss", _e)

# 7. MoE aux loss
_section("7. MoE auxiliary loss")
try:
    _sm.train()
    _sm(_zt, _t, attention_mask=_attn)
    _mv = _sm.moe_aux_loss
    _mval = _mv.item() if isinstance(_mv, _torch.Tensor) else float(_mv)
    assert _math.isfinite(_mval)
    _ok(f"moe_aux_loss = {_mval:.6f}"); _passed += 1
except Exception as _e: _fail("MoE aux loss", _e)

# 8. Backward pass
_section("8. Backward pass")
try:
    _opt = _torch.optim.AdamW([p for p in _sm.parameters() if p.requires_grad], lr=5e-5)
    _opt.zero_grad()
    _lg3 = _sm(_zt, _t, attention_mask=_attn)
    (_mdlm(_lg3, _ids, _zt, _t, _mask_id) + _sm.moe_aux_loss).backward()
    _lg = [n for n, p in _sm.named_parameters() if ("lora_A" in n or "lora_B" in n) and p.grad is not None]
    _opt.step()
    _ok(f"backward OK — {len(_lg)} LoRA params have gradients"); _passed += 1
except Exception as _e: _fail("backward pass", _e)

# 9. Mini training loop
_section(f"9. Mini training loop ({_MAX_STEPS} steps)")
try:
    _sm.train()
    _opt2 = _torch.optim.AdamW([p for p in _sm.parameters() if p.requires_grad], lr=5e-5)
    _sched = _get_sched(_opt2, num_warmup_steps=1, num_training_steps=_MAX_STEPS)
    for _s in range(_MAX_STEPS):
        _ii = _torch.randint(1000, 20000, (_BATCH_SIZE, _SEQ_LEN))
        _tt = _ns.sample_t(_BATCH_SIZE, _DEVICE)
        _zz = _ns.noise_sequence(_ii, _tt, _mask_id)
        _opt2.zero_grad()
        _ll = _sm(_zz, _tt, attention_mask=_torch.ones(_BATCH_SIZE, _SEQ_LEN, dtype=_torch.long))
        _lo = _mdlm(_ll, _ii, _zz, _tt, _mask_id) + _sm.moe_aux_loss
        _lo.backward()
        _torch.nn.utils.clip_grad_norm_(_sm.parameters(), 1.0)
        _opt2.step(); _sched.step()
        print(f"    step {_s+1}/{_MAX_STEPS}  loss={_lo.item():.4f}  lr={_sched.get_last_lr()[0]:.2e}")
    _ok(f"all {_MAX_STEPS} steps completed without NaN/Inf"); _passed += 1
except Exception as _e: _fail("mini training loop", _e)

# 10. SUBS carry-over
_section("10. SUBS — unmasked positions carry-over")
try:
    _sm.eval()
    _t_low = _torch.full((_BATCH_SIZE,), 0.01)
    _z_low = _ns.noise_sequence(_ii, _t_low, _mask_id)
    with _torch.no_grad():
        _lg_low = _sm(_z_low, _t_low, attention_mask=_torch.ones(_BATCH_SIZE, _SEQ_LEN, dtype=_torch.long))
    _unm = (_z_low != _mask_id)
    if _unm.any():
        _match = (_lg_low.argmax(-1)[_unm] == _z_low[_unm]).float().mean().item()
        _ok(f"carry-over accuracy: {_match*100:.1f}% (expect ~100%)")
    else:
        _ok("all tokens masked at t=0.01 (unusual but OK)")
    _passed += 1
except Exception as _e: _fail("SUBS carry-over", _e)

# 11. LoRA merge
_section("11. LoRA merge")
try:
    _sm.eval()
    _sm.merge_lora()
    _lt = [n for n, p in _sm.named_parameters() if ("lora_A" in n or "lora_B" in n) and p.requires_grad]
    _ok(f"merge_lora() OK — post-merge trainable LoRA params: {len(_lt)} (expect 0)"); _passed += 1
except Exception as _e: _fail("LoRA merge", _e)

# Summary
print(f"\n{'='*60}")
print(f"  SMOKE TEST COMPLETE — {_passed}/11 PASSED")
print(f"{'='*60}\n")
del _sm  # free CPU memory
